# Regular PFN on the MF prior, overnight pilot

## Description

**Question.** How well does a regular PFN (PFNs4BO-style, `surrogate=pfn`, at `pfn_large`'s size) fit the
multi-fidelity learning-curve prior (ifBO's FT-PFN prior, `priors/mf/`) in one night? Held-out NLL per number
of hyperparameters (d = 1, 2, 3) and context size (n = 5, 20, 50), and how well it extrapolates curves to
higher fidelities. Issue #4; mandate in `research_log/tasks/2026-10-22_pfn_mf.md`.

**Setup.** `experiment=pfn_mf`: d_model 256, 8 heads, 8 layers, d_ff 1024, 1000 bins (6.6M parameters, as
`pfn_large`); inputs [config (d dims), fidelity], zero-padded to 4 columns, y = 1 − performance; 50 fidelity
levels per curve; d batch-uniform in {1, 2, 3}; n_train uniform in [1, 100]; 256 query tokens; batch 64;
AdamW + warmup-cosine, bf16. A fresh prior batch every step (ifBO presampled and stored its data instead),
from `FastMFPrior(exact_stream=false)`: MFPrior's distribution with fewer torch draws, made ahead in 8 worker
processes (`training/prefetch.py`). One night on ulysses, 2026-10-22 20:00 → 2026-10-23 07:30, schedule fitted
to the deadline; the probe gave 55 ms/step, so roughly 0.6–0.7M steps. Held-out validation every 5000 steps:
fixed MF episodes per d (seed 10,000 + d; 64 tasks × 200 query tokens; context = the first n of 50 context
tokens). One seed.

**What to look for.**
- Validation NLL against steps, per d and n: does it still fall at the end of the night? How far apart are
  the d (more hyperparameters, less data per curve region) and the n?
- Afterwards, from the checkpoint (descriptive, not part of the rule): NLL of queries that continue an observed
  curve, by the gap between the query fidelity and the curve's last observed fidelity, against queries of
  curves not seen in the context. Continuations should be much sharper than new curves; the question is how
  fast the sharpness decays with the gap.
- Stability: no NaN/inf, no loss spikes under bf16.

**GO / NO-GO** (for a longer run of this configuration; on `nll/val_d{d}_n50`). Drop = the value at the
validation point closest to 75% of the steps minus the final value, averaged over d = 1, 2, 3:
- GO if the drop is ≥ 0.02 nats (it keeps falling at the end of the night);
- NO-GO if the drop is < 0.005 nats (flat: a longer run of this configuration is unlikely to pay off);
- otherwise INCONCLUSIVE.

**Limits.**
- The cosine schedule anneals the learning rate to 0 at the deadline, so part of any late flattening is
  the schedule, not saturation: the rule leans towards NO-GO.
- One seed, one size; 50 fidelity levels fixed (`prior.n_levels`), at most 3 hyperparameters, n_train ≤ 100:
  the checkpoint is valid inside these ranges only.
- The validation sets come from the same prior: this measures fit to the prior, not performance on real
  learning curves (e.g. LCBench), which is a separate evaluation.

In [ ]:
from pfn_repro.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/pfn_mf/<sweep_id>"  # set after `just snapshot runs/pfn_mf/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.